In [ ]:
import model
import epidemic_simulation
import survey_design
import random
import numpy as np
import scipy.stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import arviz
import cmdstanpy
from cmdstanpy import CmdStanModel

In [ ]:
random.seed(123)
np.random.seed(123)

In [ ]:
f_infectious, f_recovered, delay_inf, delay_recov = epidemic_simulation.delays()
transmission_rate = epidemic_simulation.transmission_rate

In [ ]:
inference_model = CmdStanModel(stan_file='model_rw2.stan', cpp_options={'STAN_THREADS':'true'})

sample_sizes = [100, 200, 400, 800,]
prev_sample_sizes = np.repeat(sample_sizes, len(sample_sizes))
sero_sample_sizes = np.tile(sample_sizes, len(sample_sizes))

spacings = [1] * len(prev_sample_sizes)

num_repeats = 10

dfs = []
all_scores = []
all_fits = []
all_expanded_scores = []
all_scores_energy = []
all_coverages = []
all_widths = []


In [ ]:
for _ in range(num_repeats):

    m = model.AgentModel(1000000, 50)

    surveys = survey_design.make_surveys(m, spacings, prev_sample_sizes, spacings, sero_sample_sizes, duration=1)
    df = survey_design.run_simulations(m, surveys, transmission_rate, f_infectious, f_recovered)

    fits = survey_design.run_inference(df, surveys, inference_model, delay_inf, delay_recov, 'rolling', init_mcmc=800)
    scores, expanded_scores, scores_energy, coverage, width = survey_design.evaluate_fits(df, fits)

    dfs.append(df)
    all_scores.append(scores)
    all_fits.append(fits)
    all_expanded_scores.append(expanded_scores)

    all_scores_energy.append(scores_energy)
    all_coverages.append(coverage)
    all_widths.append(width)


In [ ]:
all_scores = np.asarray(all_scores)
avg_scores = np.median(all_scores, axis=0)

all_scores_energy = np.asarray(all_scores_energy)
avg_scores_energy = np.median(all_scores_energy, axis=0)

In [ ]:
fig = plt.figure(figsize=(5, 3.5))
ax = fig.add_subplot(1, 1, 1)

dense_ss_prev = np.arange(80, 1000, 2)
dense_ss_sero = np.arange(80, 1000, 2)

X, Y = np.meshgrid(dense_ss_prev, dense_ss_sero)

effort = np.zeros(X.shape)
for i, prev_ss in enumerate(dense_ss_prev):
    for j, sero_ss in enumerate(dense_ss_sero):
        surveys = survey_design.make_surveys(m, [1], [prev_ss], [1], [sero_ss])
        effort[j, i] = surveys[0].sample_size * len(surveys[0].start_dates) + surveys[1].sample_size * len(surveys[1].start_dates)

levels = [80000, 160000, 240000, 320000, 400000, 480000]

true_levels = []
for l in levels:
    nearest = min(effort.flatten(), key=lambda x:abs(x-l))
    true_levels.append(nearest)

cs = ax.contour(X, Y, effort, true_levels, colors='tab:red', zorder=-10,)
ax.clabel(cs, cs.levels, fontsize=10)


cmap = mpl.colormaps['Greys_r']

highest_crps = max(avg_scores)
lowest_crps = min(avg_scores)

for score, prev_sample_size, sero_sample_size in zip(avg_scores, prev_sample_sizes, sero_sample_sizes):
    color = cmap((score - lowest_crps) / (highest_crps - lowest_crps))
    ax.scatter([prev_sample_size,], [sero_sample_size,], color=color, zorder=-1, s=300)
    ax.scatter([prev_sample_size,], [sero_sample_size,], color='k', zorder=-2, alpha=1, s=400)
    # ax.text(prev_sample_size, sero_sample_size, '{:.2f}'.format(score), color='red', zorder=100, ha='left')

cbar = plt.colorbar(cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=lowest_crps, vmax=highest_crps), cmap=cmap), ax=ax, label='Score')
cbar.set_ticks([lowest_crps, highest_crps])
cbar.set_ticklabels(['Best', 'Worst'])

ax.set_xlabel('Daily tests infection prevalence')
ax.set_ylabel('Daily tests seroprevalence')

ax.set_yscale('log')
ax.set_xscale('log')

ax.set_xticks(sample_sizes)
ax.set_yticks(sample_sizes)
ax.set_xticklabels(sample_sizes)
ax.set_yticklabels(sample_sizes)

ax.set_xticks([], minor=True)
ax.set_yticks([], minor=True)

fig.set_tight_layout(True)

plt.savefig('Figure6a.pdf')

plt.show()


In [ ]:
fig = plt.figure(figsize=(4.5, 2.5))
ax = fig.add_subplot(1, 1, 1)

for ss in sample_sizes:
    sero_scores = all_scores[:, np.where(sero_sample_sizes == ss)]
    mid = np.mean(sero_scores, axis=0)[0]
    ax.plot(sample_sizes, mid, '.-', label=ss)
    # std = np.std(sero_scores, axis=0)[0]
    # ax.fill_between(spacings, mid - std, mid + std, alpha=0.25)

# ax.set_xscale('log')
ax.spines[['right', 'top']].set_visible(False)
ax.legend(title='Sero. Sample size', loc='center left', bbox_to_anchor=(1, 0.5))

ax.set_xlabel('Infection Sample size')
ax.set_ylabel('Score')

fig.set_tight_layout(True)

plt.show()

In [ ]:
sample_sizes

In [ ]:
all_widths = np.asarray(all_widths)

fig = plt.figure(figsize=(6, 3.25))

ax = fig.add_subplot(1, 1, 1)

colors = {100: 'tab:blue', 200: 'tab:orange', 400: 'tab:green', 800: 'tab:red'}

sample_sizes_to_consider = {21: (0, -1), 42: (1, None), 84: (2, None)}
scores_plotted = []

for j, prev_ss in enumerate(sample_sizes):
    spacing_scores = all_scores[:, np.where(prev_sample_sizes == prev_ss)]
    mid = np.median(spacing_scores, axis=0)[0]

    # spacing_scores = all_scores[1, np.where(full_spacings == spacing)]
    # mid = np.mean(spacing_scores, axis=0)

    # s0, s1 = sample_sizes_to_consider[spacing]
    # mid = mid[s0:s1]
    # for i, sample_size in enumerate(sample_sizes[s0:s1]):
    for i, sero_ss in enumerate(sample_sizes):
        surveys = survey_design.make_surveys(m, [1], [prev_ss], [1], [sero_ss], duration=7)
        effort = surveys[0].sample_size * len(surveys[0].start_dates) + surveys[1].sample_size * len(surveys[1].start_dates)

        # ax.scatter([effort,], [mid[i],], color=colors[spacing], s=50, label=spacing if i == 0 else None)
        for k in range(spacing_scores.shape[0]):
            ax.scatter([effort,], [spacing_scores[k, 0, i],], color=colors[prev_ss], s=25, label=prev_ss if k == 0 and i == 0 else None, alpha=0.5)
            scores_plotted.append(spacing_scores[k, 0, i])


ax.legend(title='Inf. prev. samples\nper day', loc='center left', bbox_to_anchor=(1, 0.5))

yticks = [np.min(scores_plotted), np.max(scores_plotted)]
ax.set_yticks(yticks)
ax.set_yticklabels(['Best', 'Worst'])

ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Total tests conducted')
ax.set_ylabel('Score')

# ax.set_xscale('log')
# ax.set_yscale('log')

# ax.ticklabel_format(style='plain', axis='x')
plt.xticks(rotation=90, ha='center')

fig.set_tight_layout(True)

plt.savefig('Figure6b.pdf')

# ax.set_ylim(500, 17000)

plt.show()

In [ ]:
surveys = survey_design.make_surveys(m, spacings, prev_sample_sizes, spacings, sero_sample_sizes)

fig = plt.figure(figsize=(7, 2.75), dpi=512)

idx1 = 3
idx2 = 15
idx3 = 0
idx4 = 12

dfi = 0
df = dfs[dfi]

ax = fig.add_subplot(2, 2, 1)

fit = all_fits[dfi][idx1]


ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

ax.set_ylabel('Incidence')

ax.set_title('100 infection\ntests per day')

ax.tick_params(labelbottom=False, labelleft=True)



ax = fig.add_subplot(2, 2, 2, sharex=ax, sharey=ax)

fit = all_fits[dfi][idx2]



ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)

legend_ax = ax

ax.set_title('800 infection\ntests per day')


ax.tick_params(labelbottom=False, labelleft=False)


ax = fig.add_subplot(2, 2, 3, sharex=ax, sharey=ax)

fit = all_fits[dfi][idx3]



ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

ax.plot(tplot, mid,)
ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)


ax.set_ylabel('Incidence')
ax.set_xlabel('Time (days)')


ax.set_title('100 sero.\ntests per day', x=-0.5, y=0.4)


ax = fig.add_subplot(2, 2, 4, sharex=ax, sharey=ax)

fit = all_fits[dfi][idx4]



l6, = ax.plot(df['transmissions'], color='k', label='Incidence')

x = np.asarray(fit.incidence[:, :])
lower = np.percentile(x, 5, axis=0,)
upper = np.percentile(x, 95, axis=0,)
mid = np.median(x, axis=0)
tplot = np.arange(len(mid))

l4, = ax.plot(tplot, mid,)
l5 = ax.fill_between(tplot, lower, upper, alpha=0.35)
ax.spines[['right', 'top']].set_visible(False)
ax.set_xlabel('Time (days)')

ax.set_title('800 sero.\ntests per day', x=-1.7, y=1.6)

legend_ax.legend(((l4, l5,), l6,), ['Posterior', 'Incidence'], loc='center left', bbox_to_anchor=(1, 0.5))

ax.tick_params(labelbottom=True, labelleft=False)

fig.set_tight_layout(True)

# plt.savefig('Figure6b.pdf')

plt.show()